In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()
print(f"Loaded model: {model_name}")
print(f"Hidden size: {model.config.hidden_size}")


In [ ]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])


In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_texts(texts, batch_size=64, max_length=128):
    all_embeddings = []
    for start_idx in range(0, len(texts), batch_size):
        batch_texts = texts[start_idx:start_idx + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

sentence1_list = dataset["sentence1"]
sentence2_list = dataset["sentence2"]
labels = dataset["label"]

emb1 = encode_texts(sentence1_list, batch_size=64, max_length=128)
emb2 = encode_texts(sentence2_list, batch_size=64, max_length=128)

cosine_similarities = F.cosine_similarity(emb1, emb2).tolist()

threshold = 0.80
predictions = [1 if score >= threshold else 0 for score in cosine_similarities]
confidences = [abs(score - threshold) for score in cosine_similarities]

print(f"Completed embedding inference for {len(predictions)} examples.")
print(f"Fixed cosine similarity threshold: {threshold}")


In [ ]:
accuracy = accuracy_score(labels, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
cm = confusion_matrix(labels, predictions)

positive_scores = [score for score, label in zip(cosine_similarities, labels) if label == 1]
negative_scores = [score for score, label in zip(cosine_similarities, labels) if label == 0]
mean_positive_similarity = sum(positive_scores) / len(positive_scores)
mean_negative_similarity = sum(negative_scores) / len(negative_scores)

print("Evaluation metrics:")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1       : {f1:.4f}")
print("Confusion matrix:")
print(cm)
print(f"Mean cosine similarity | label=1: {mean_positive_similarity:.4f}")
print(f"Mean cosine similarity | label=0: {mean_negative_similarity:.4f}")


In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
num_examples_to_show = 8

ranked_indices = sorted(range(len(dataset)), key=lambda i: abs(cosine_similarities[i] - threshold))

for rank, i in enumerate(ranked_indices[:num_examples_to_show], start=1):
    row = dataset[i]
    true_label = labels[i]
    pred_label = predictions[i]
    score = cosine_similarities[i]
    margin = abs(score - threshold)
    print(f"Near-threshold example {rank}")
    print(f"index: {i}")
    print(f"sentence1: {row['sentence1']}")
    print(f"sentence2: {row['sentence2']}")
    print(f"true label: {true_label} ({label_map[true_label]})")
    print(f"pred label: {pred_label} ({label_map[pred_label]})")
    print(f"cosine similarity: {score:.4f}")
    print(f"distance from threshold: {margin:.4f}")
    print("-" * 80)


In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("inference_method=separate_sentence_embeddings_with_cosine_similarity")
print("dataset_split=glue/mrpc validation")
print(f"device={device}")
print(f"num_examples={len(dataset)}")
print(f"threshold={threshold}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"mean_positive_similarity={mean_positive_similarity:.4f}")
print(f"mean_negative_similarity={mean_negative_similarity:.4f}")
